# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 mass spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema accessible at the following URL:

In [ ]:
# Ensure the latest mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
We load the Croissant metadata and records using the `mlcroissant` API.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Cite as: {getattr(dataset.metadata, 'citeAs', None)}")


## 2. Data Overview
Let's review available record sets, their fields, column names, and their `@id` values. All will be referenced by their `@id`.

In [ ]:
# List record sets
print("Record Sets in the dataset:")
record_sets = dataset.metadata.recordSet
# If record_sets is None, try 'recordSet' as a list
if record_sets is None:
    record_sets = []

for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', None)} | description: {rs.get('description', None)}")
    print("  Fields:")
    for field in rs.get('field', []):
        print(f"    - {field['@id']}: {field.get('name', '')}")

### Example: Print records from a record set
Here we retrieve and print the first 2 records using their `@id` (replace with a real record set `@id` if discovered above).


In [ ]:
# For demonstration, let's list all record set @ids
record_set_ids = [rs['@id'] for rs in getattr(dataset.metadata, 'recordSet', [])]
print('Available record set @id values:', record_set_ids)

# Example: Print first 2 records in the first record set
if record_set_ids:
    chosen_record_set_id = record_set_ids[0]
    for i, rec in enumerate(dataset.records(record_set=chosen_record_set_id)):
        print(rec)
        if i >= 1:
            break
else:
    print("No record sets found in the metadata.")


## 3. Data Extraction
We will load all tabular data from each available record set into a pandas DataFrame, referenced by record set `@id`. This forms the basis for analysis. You can later reference columns by field or column `@id` (see previous outputs).

In [ ]:
# Store all dataframes by record set @id
dataframes = {}

record_set_ids = [rs['@id'] for rs in getattr(dataset.metadata, 'recordSet', [])]
for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for {record_set_id}.")

# For downstream analysis, choose the main record set (update as appropriate)
if dataframes:
    primary_record_set_id = list(dataframes.keys())[0]
    print(f"Using {primary_record_set_id} for further analysis.")
    print("Columns:", dataframes[primary_record_set_id].columns.tolist())
else:
    print("No tabular record sets extracted.")

## 4. Exploratory Data Analysis (EDA)
Let's filter and process numeric fields. All operations reference columns by string based on their field `@id` or logical column names. Update the field id as appropriate.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

if dataframes:
    df = dataframes[primary_record_set_id]
    # Try to auto-detect a numeric field
    numeric_candidates = []
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_candidates.append(col)
    print(f"Numeric field candidates: {numeric_candidates}")

    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # You could change to your field of interest
    else:
        # Try to coerce if all columns are string/object
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
                if np.issubdtype(df[col].dtype, np.number):
                    numeric_field = col
                    break
            except Exception:
                continue

    if 'numeric_field' in locals():
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold:.2f}")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a grouping/categorical field
        potential_cat = [c for c in df.columns if c != numeric_field and df[c].nunique() < len(df)/4]
        group_field = potential_cat[0] if potential_cat else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No suitable numeric field detected for EDA.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Let's visualize the distribution of the chosen numeric field or its normalized version. If a categorical grouping field is available, plot group means as well.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    if 'filtered_df' in locals() and f"{numeric_field}_normalized" in filtered_df:
        plt.figure(figsize=(8,5))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True, color='orange')
        plt.title(f'Normalized {numeric_field} (records above mean)')
        plt.xlabel(f"{numeric_field}_normalized")
        plt.show()

    if 'group_field' in locals() and group_field:
        fig, ax = plt.subplots(figsize=(10,4))
        grouped = filtered_df.groupby(group_field)[numeric_field].mean()
        grouped.plot(kind='bar', ax=ax)
        plt.title(f'Mean {numeric_field} by {group_field} (filtered records)')
        plt.ylabel(numeric_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 mass spectrometry dataset using the Croissant schema, explored the data structure and available fields, and performed basic numeric EDA by filtering and normalizing values. Visualizations were generated to inspect numeric distributions and group means.

- For in-depth analysis, explore the full Croissant metadata and the record sets/fields it contains. 
- All operations can be referenced by Croissant `@id` for reproducibility and interoperability.

_For more advanced usage and integration, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/)_